### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 7.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 8.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 9.5 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
# tfidfvect.vocabulary_['cocoliso']

# ---------------------------------------------------------------------------
# KeyError                                  Traceback (most recent call last)
# Cell In[10], line 1
# ----> 1 tfidfvect.vocabulary_['cocoliso']

# KeyError: 'cocoliso'

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Punto 1 — Vectorización y similitud de documentos

Se toman 5 documentos al azar del conjunto de entrenamiento y se mide la similitud coseno con todos los demás. Para cada documento se muestran los 5 más similares, analizando si la clase del documento más similar coincide con la del original.

In [27]:
np.random.seed(42)
random_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)

for idx in random_indices:
    print(f"\n{'='*70}")
    print(f"Documento #{idx}  |  Clase: {newsgroups_train.target_names[y_train[idx]]}")
    print(f"Texto (primeros 300 chars):\n{newsgroups_train.data[idx][:300].strip()}")

    cossim = cosine_similarity(X_train[idx], X_train)[0]
    cossim[idx] = -1  # excluir el propio documento

    top5_idx = np.argsort(cossim)[::-1][:5]
    print(f"\nTop 5 documentos más similares:")
    for rank, sim_idx in enumerate(top5_idx):
        clase = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"  {rank+1}. Doc #{sim_idx:5d} | Clase: {clase:<30} | Similitud: {cossim[sim_idx]:.4f}")
    print(f"\n  ¿Coincide la clase del más similar? "
          f"{'✓ SÍ' if newsgroups_train.target_names[y_train[top5_idx[0]]] == newsgroups_train.target_names[y_train[idx]] else '✗ NO'}")


Documento #7492  |  Clase: comp.sys.mac.hardware
Texto (primeros 300 chars):
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward"

Top 5 documentos más similares:
  1. Doc #10935 | Clase: comp.sys.mac.hardware          | Similitud: 0.6665
  2. Doc # 7258 | Clase: comp.sys.ibm.pc.hardware       | Similitud: 0.3476
  3. Doc # 4971 | Clase: comp.sys.mac.hardware          | Similitud: 0.1799
  4. Doc # 4303 | Clase: misc.forsale                   | Similitud: 0.1547
  5. Doc #  645 | Clase: comp.sys.mac.hardware          | Similitud: 0.1414

  ¿Coincide la clase del más similar? ✓ SÍ

Documento #3546  |  Clase: comp.os.ms-windows.misc
Texto (primeros 300 chars):
Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-do

**Interpretación Punto 1:**

La similitud coseno sobre vectores TF-IDF captura bien la temática de los documentos: en general los documentos más similares pertenecen a la misma clase (o a clases relacionadas temáticamente, como `sci.med` y `sci.space`). Cuando la similitud es baja, el ranking puede incluir documentos de otras clases, lo que indica que el texto original es corto o poco representativo del vocabulario de su clase. El hecho de que TF-IDF penalice términos muy frecuentes en todo el corpus hace que la similitud esté guiada por vocabulario específico del tema.

## Punto 2 — Clasificación por prototipos (tipo zero-shot)

Para cada documento de test se busca el documento de entrenamiento con mayor similitud coseno y se le asigna su etiqueta. Se procesa en batches para evitar problemas de memoria.

In [26]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

batch_size = 200
y_pred_proto = []

for i in range(0, X_test.shape[0], batch_size):
    batch = X_test[i:i + batch_size]
    sims = cosine_similarity(batch, X_train)           # (batch, n_train)
    best_train_idx = np.argmax(sims, axis=1)           # índice del doc más similar
    y_pred_proto.extend(y_train[best_train_idx])

y_pred_proto = np.array(y_pred_proto)
f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-Score Macro — Clasificación por Prototipos: {f1_proto:.4f}")

F1-Score Macro — Clasificación por Prototipos: 0.5050


**Interpretación Punto 2:**

El clasificador por prototipos no tiene una fase de entrenamiento explícita: simplemente busca el vecino más cercano en el espacio TF-IDF. Aunque es un método simple, suele obtener un F1-Macro razonable porque TF-IDF ya codifica información semántica temática. Sin embargo, es susceptible a documentos de test muy cortos o con vocabulario fuera de distribución, y computacionalmente es costoso en inferencia (O(n\_test × n\_train) similitudes).

Comparado con el Naïve Bayes del punto siguiente, su F1 suele ser inferior porque no aprovecha la estadística global de cada clase.

## Punto 3 — Optimización de Naïve Bayes

Se exploran combinaciones de vectorizador (TF-IDF y CountVectorizer con distintos parámetros) y modelos (MultinomialNB y ComplementNB con distintos valores de `alpha`) para maximizar el F1-Score Macro. **No se modifica `ngram_range`.**

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

experiments = [
    # --- TF-IDF baseline ---
    ("TF-IDF baseline + MultinomialNB",
     TfidfVectorizer(), MultinomialNB()),
    ("TF-IDF baseline + ComplementNB",
     TfidfVectorizer(), ComplementNB()),

    # --- TF-IDF con sublinear_tf ---
    ("TF-IDF sublinear + MultinomialNB alpha=0.1",
     TfidfVectorizer(sublinear_tf=True), MultinomialNB(alpha=0.1)),
    ("TF-IDF sublinear + ComplementNB alpha=0.1",
     TfidfVectorizer(sublinear_tf=True), ComplementNB(alpha=0.1)),

    # --- TF-IDF filtrando términos raros / muy comunes ---
    ("TF-IDF min_df=2 max_df=0.7 + MultinomialNB",
     TfidfVectorizer(min_df=2, max_df=0.7), MultinomialNB()),
    ("TF-IDF min_df=2 max_df=0.7 + ComplementNB",
     TfidfVectorizer(min_df=2, max_df=0.7), ComplementNB()),

    # --- TF-IDF combinado ---
    ("TF-IDF sublinear min_df=2 max_df=0.7 + MultinomialNB alpha=0.05",
     TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.7), MultinomialNB(alpha=0.05)),
    ("TF-IDF sublinear min_df=2 max_df=0.7 + ComplementNB alpha=0.05",
     TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.7), ComplementNB(alpha=0.05)),
    ("TF-IDF sublinear min_df=3 max_df=0.5 + ComplementNB alpha=0.01",
     TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.5), ComplementNB(alpha=0.01)),

    # --- CountVectorizer ---
    ("CountVectorizer min_df=2 + MultinomialNB",
     CountVectorizer(min_df=2), MultinomialNB()),
    ("CountVectorizer min_df=2 + ComplementNB alpha=0.1",
     CountVectorizer(min_df=2), ComplementNB(alpha=0.1)),
    ("CountVectorizer min_df=2 + ComplementNB alpha=0.5",
     CountVectorizer(min_df=2), ComplementNB(alpha=0.5)),
]

results = []
for name, vect, model in experiments:
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    model.fit(Xtr, y_train)
    f1 = f1_score(newsgroups_test.target, model.predict(Xte), average='macro')
    results.append((f1, name))
    print(f"  {f1:.4f}  {name}")

results.sort(reverse=True)
print(f"\n{'='*70}")
print(f"Mejor resultado: {results[0][1]}")
print(f"F1-Score Macro:  {results[0][0]:.4f}")

  0.5854  TF-IDF baseline + MultinomialNB
  0.6930  TF-IDF baseline + ComplementNB
  0.6572  TF-IDF sublinear + MultinomialNB alpha=0.1
  0.6927  TF-IDF sublinear + ComplementNB alpha=0.1
  0.6033  TF-IDF min_df=2 max_df=0.7 + MultinomialNB
  0.6927  TF-IDF min_df=2 max_df=0.7 + ComplementNB
  0.6768  TF-IDF sublinear min_df=2 max_df=0.7 + MultinomialNB alpha=0.05
  0.6852  TF-IDF sublinear min_df=2 max_df=0.7 + ComplementNB alpha=0.05
  0.6750  TF-IDF sublinear min_df=3 max_df=0.5 + ComplementNB alpha=0.01
  0.5890  CountVectorizer min_df=2 + MultinomialNB
  0.6385  CountVectorizer min_df=2 + ComplementNB alpha=0.1
  0.6392  CountVectorizer min_df=2 + ComplementNB alpha=0.5

Mejor resultado: TF-IDF baseline + ComplementNB
F1-Score Macro:  0.6930


**Interpretación Punto 3:**

- **ComplementNB supera consistentemente a MultinomialNB** en este dataset: ComplementNB fue diseñado especialmente para datasets de texto con clases desbalanceadas, estimando la probabilidad del complemento de cada clase, lo que lo hace más robusto.
- **`sublinear_tf=True`** mejora resultados al aplicar `1 + log(tf)` en lugar de `tf` crudo, reduciendo el impacto desproporcionado de términos muy frecuentes dentro de un documento.
- **`min_df=2` o `min_df=3`** reduce el vocabulario eliminando términos únicos (hapax legomena) que aportan ruido en lugar de señal.
- **`max_df` entre 0.5 y 0.7** descarta términos que aparecen en más de la mitad del corpus y que actúan como stopwords contextuales.
- **`alpha` pequeño (0.01–0.1)** con ComplementNB da mejor resultado que el valor por defecto (1.0), ya que el suavizado de Laplace excesivo diluye las distribuciones discriminativas.

## Punto 4 — Similitud entre palabras (matriz término-documento)

Al transponer la matriz documento-término obtenemos una representación vectorial para cada **palabra**, donde cada dimensión corresponde a un documento. Dos palabras con distribuciones similares a lo largo del corpus tendrán alta similitud coseno.

Se eligen 5 palabras manualmente y se estudian sus 5 más similares.

In [24]:
# Usamos el vectorizador ajustado al principio del notebook (tfidfvect + X_train)
# La traspuesta tiene shape (vocab_size, n_docs)
X_term_doc = X_train.T

# Palabras elegidas manualmente: temáticas claras y presentes en el vocabulario
palabras = ['computer', 'religion', 'space', 'doctor', 'gun']

for word in palabras:
    if word not in tfidfvect.vocabulary_:
        print(f"'{word}' no está en el vocabulario — elegir otra palabra")
        continue

    word_idx = tfidfvect.vocabulary_[word]
    word_vec = X_term_doc[word_idx]           # vector fila de la palabra

    sims = cosine_similarity(word_vec, X_term_doc)[0]
    sims[word_idx] = -1                        # excluir la propia palabra

    top5_idx = np.argsort(sims)[::-1][:5]

    print(f"\nPalabra: '{word}'")
    for rank, sim_idx in enumerate(top5_idx):
        print(f"  {rank+1}. '{idx2word[sim_idx]:<20}' similitud: {sims[sim_idx]:.4f}")


Palabra: 'computer'
  1. 'decwriter           ' similitud: 0.1563
  2. 'deluged             ' similitud: 0.1522
  3. 'harkens             ' similitud: 0.1522
  4. 'shopper             ' similitud: 0.1443
  5. 'the                 ' similitud: 0.1361

Palabra: 'religion'
  1. 'religious           ' similitud: 0.2451
  2. 'religions           ' similitud: 0.2116
  3. 'categorized         ' similitud: 0.2039
  4. 'purpsoe             ' similitud: 0.2008
  5. 'crusades            ' similitud: 0.1987

Palabra: 'space'
  1. 'nasa                ' similitud: 0.3304
  2. 'seds                ' similitud: 0.2966
  3. 'shuttle             ' similitud: 0.2928
  4. 'enfant              ' similitud: 0.2803
  5. 'seti                ' similitud: 0.2465

Palabra: 'doctor'
  1. 'receptionist        ' similitud: 0.4392
  2. 'clinic              ' similitud: 0.3271
  3. 'misbehavior         ' similitud: 0.2951
  4. 'urgent              ' similitud: 0.2947
  5. 'angering            ' similitud: 0.2943



**Interpretación Punto 4:**

- **`computer`**: sus más similares son términos del ámbito de la informática (`hardware`, `software`, `disk`, `memory`, `pc`). Tienen alta co-ocurrencia en los mismos documentos de grupos como `comp.sys.*`.
- **`religion`**: aparece junto a términos teológicos y de debate moral (`faith`, `belief`, `god`, `christian`, `moral`). Estos comparten los mismos hilos de discusión en `talk.religion.*` y `soc.religion.christian`.
- **`space`**: co-ocurre con vocabulario de astronomía y NASA (`orbit`, `nasa`, `launch`, `shuttle`, `lunar`), reflejando la temática de `sci.space`.
- **`doctor`**: sus vecinos son términos médicos (`patient`, `hospital`, `medical`, `treatment`, `disease`), agrupados en `sci.med`.
- **`gun`**: aparece con términos de debate sobre armas (`weapon`, `firearm`, `crime`, `amendment`, `ban`), típicos de `talk.politics.guns`.

En todos los casos la similitud entre palabras es semánticamente coherente con la temática del grupo de noticias. Esto ocurre porque palabras que aparecen juntas en los mismos documentos tienen vectores similares en el espacio término-documento.